# NxN Grid Visualizer

Each cell in the grid is rendered as an **MxM pixel square**, colored either **yellow** or **blue** based on a random draw against the yellow probability `p_yellow`. Blue probability is automatically computed as `p_blue = 1 - p_yellow`.

Use the sliders below to configure the grid.

In [4]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display

In [5]:
def generate_grid(N: int, p_yellow: float) -> list[list[str]]:
    """Generate an NxN grid where each cell is 'yellow' or 'blue'.
    
    Args:
        N: Grid dimension (NxN cells).
        p_yellow: Probability [0, 1] that a cell is yellow.
    
    Returns:
        2D list of color strings.
    """
    return [
        ['yellow' if random.random() < p_yellow else 'blue' for _ in range(N)]
        for _ in range(N)
    ]


def draw_grid(grid: list[list[str]], M: int, ax: plt.Axes) -> None:
    """Draw the grid onto a matplotlib Axes object.
    
    Args:
        grid: 2D list of color strings ('yellow' or 'blue').
        M: Pixel size of each cell square.
        ax: Matplotlib axes to draw on.
    """
    N = len(grid)
    ax.set_xlim(0, N * M)
    ax.set_ylim(0, N * M)
    ax.set_aspect('equal')
    ax.axis('off')

    for row in range(N):
        for col in range(N):
            color = grid[row][col]
            # Invert row so row 0 appears at the top
            x = col * M
            y = (N - 1 - row) * M
            rect = mpatches.Rectangle(
                (x, y), M, M,
                linewidth=0.5,
                edgecolor='white',
                facecolor=color
            )
            ax.add_patch(rect)

In [ ]:
# ── Widgets ──────────────────────────────────────────────────────────────────

n_slider = widgets.IntSlider(
    value=10, min=2, max=500, step=1,
    description='N (grid size):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

m_slider = widgets.IntSlider(
    value=30, min=1, max=80, step=1,
    description='M (cell pixels):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

p_slider = widgets.FloatSlider(
    value=0.5, min=0.0, max=1.0, step=0.01,
    description='p(yellow):',
    readout_format='.2f',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

p_blue_label = widgets.Label(value='p(blue) = 0.50')

regenerate_btn = widgets.Button(
    description='Regenerate',
    button_style='primary',
    icon='refresh'
)

output = widgets.Output()

# ── Callbacks ─────────────────────────────────────────────────────────────────

def update_blue_label(change: dict) -> None:
    """Keep the p(blue) label in sync with the yellow slider."""
    p_blue_label.value = f'p(blue) = {1 - change["new"]:.2f}'

p_slider.observe(update_blue_label, names='value')


def render(_=None) -> None:
    """Regenerate and redraw the grid."""
    N = n_slider.value
    M = m_slider.value
    p_yellow = p_slider.value
    p_blue = 1 - p_yellow

    grid = generate_grid(N, p_yellow)

    # Count actual proportions
    total = N * N
    n_yellow = sum(cell == 'yellow' for row in grid for cell in row)
    n_blue = total - n_yellow

    fig_size = max(4, N * M / 72)  # rough inch estimate at 72 dpi
    fig, ax = plt.subplots(figsize=(fig_size, fig_size), dpi=72)
    draw_grid(grid, M, ax)

    fig.suptitle(
        f'N={N}  |  M={M}px  |  '
        f'p(yellow)={p_yellow:.2f}  p(blue)={p_blue:.2f}\n'
        f'Actual: {n_yellow}/{total} yellow ({n_yellow/total:.1%}), '
        f'{n_blue}/{total} blue ({n_blue/total:.1%})',
        fontsize=10
    )
    plt.tight_layout()

    with output:
        output.clear_output(wait=True)
        plt.show()
    plt.close(fig)


regenerate_btn.on_click(render)

# ── Layout ────────────────────────────────────────────────────────────────────

controls = widgets.VBox([
    widgets.HTML('<b>Grid Configuration</b>'),
    n_slider,
    m_slider,
    widgets.HBox([p_slider, p_blue_label]),
    regenerate_btn
])

display(controls, output)
render()  # Draw initial grid

Output()